In [63]:
import pandas as pd
import numpy as np
import random
from datetime import datetime , timedelta
from sklearn.preprocessing import LabelEncoder as le
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.linear_model import LinearRegression as lr
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn import tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
from sklearn.utils import shuffle
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [64]:
df = pd.read_excel("Online Retail.xlsx")

In [65]:
# --- 1.1 — Profile the raw data ---
print("Number of rows and columns" , df.shape)
print("Memory usage" , df.memory_usage()) 
for i in df.columns:
    print(f"Counts of missing values of {i}" , df[i].isnull().sum())
    print(f"Percentages missing values {i}" , (df[i].isnull().sum() / len(df)) * 100)
    print(f"Number of unique values per column {i}" , df[i].nunique())
    numeric_cols_df = df.select_dtypes(include=[np.number])
    
for a in numeric_cols_df.columns:
    print("Min" , numeric_cols_df[a].min())
    print("Max" , numeric_cols_df[a].max())
    print("Mean" , numeric_cols_df[a].mean())
    print("Median" , numeric_cols_df[a].median())
    print("Std" , numeric_cols_df[a].std())

Number of rows and columns (541909, 8)
Memory usage Index              132
InvoiceNo      4335272
StockCode      4335272
Description    4335272
Quantity       4335272
InvoiceDate    4335272
UnitPrice      4335272
CustomerID     4335272
Country        4335272
dtype: int64
Counts of missing values of InvoiceNo 0
Percentages missing values InvoiceNo 0.0
Number of unique values per column InvoiceNo 25900
Counts of missing values of StockCode 0
Percentages missing values StockCode 0.0
Number of unique values per column StockCode 4070
Counts of missing values of Description 1454
Percentages missing values Description 0.2683107311375157
Number of unique values per column Description 4223
Counts of missing values of Quantity 0
Percentages missing values Quantity 0.0
Number of unique values per column Quantity 722
Counts of missing values of InvoiceDate 0
Percentages missing values InvoiceDate 0.0
Number of unique values per column InvoiceDate 23260
Counts of missing values of UnitPrice 0
Perce

TASK 1.2 — Classify the missing data types

I split the data into rows with missing CustomerID and rows with valid CustomerID and compared their statistics using the describe() method. The groups show noticeable differences, suggesting that the missingness depends on other variables in the dataset. Therefore, the missingness is likely MAR.

For Description column, i dropped nan values because it is a few rows with nan values. But we can't to this to CustomerID because nan values holds 25% of our total data.
It is difficult to recover missing descriptions using StockCode because the same StockCode can appear with different descriptions in the dataset.

In [66]:
# --- 1.3 — Handle the missing values --- 
print(df.isnull().sum())
df["Description"] = df["Description"].fillna("Unknown Description")
most_common_id = df.groupby("Country")["CustomerID"].agg(
    lambda x: x.value_counts().idxmax() if not x.value_counts().empty else np.nan
)

df["CustomerID"] = df["CustomerID"].fillna(df["Country"].map(most_common_id))
df.loc[(df["CustomerID"].isnull()) & (df["Country"] == "Hong Kong"), "CustomerID"] = 8
print(df.isnull().sum())
df

# I handled missing values as follows:
# - For Description, I replaced missing values with "Unknown Description" because it is a categorical text field.
# - For CustomerID, I filled missing values using the most frequent CustomerID within the same Country.
# - For Hong Kong, no valid CustomerID was available in that country group, so I assigned a placeholder value of 8.
# Finally, I printed the remaining missing value counts to confirm that missing values were handled.

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


Task 1 — Data Profiling and Missing Value Handling
1.1 Profiling the Raw Data

First, a basic profile of the dataset was created to understand its structure and identify potential data quality issues.

The following checks were performed:

The number of rows and columns was obtained using df.shape.

Memory usage was inspected to understand the dataset size.

For each column, the number and percentage of missing values were calculated.

The number of unique values per column was computed to understand the diversity of the data.

For numeric columns, basic statistics such as minimum, maximum, mean, median, and standard deviation were calculated to understand the distribution of numeric variables.

This profiling step helped identify that CustomerID and Description contained a significant number of missing values.

1.2 Classification of Missing Data Types

The two columns with significant missing values are CustomerID and Description.

CustomerID

To analyze the missingness pattern, the dataset was divided into two groups:

rows with missing CustomerID

rows with valid CustomerID

The statistics of both groups were compared using the describe() method. The comparison showed noticeable differences in the distributions of other variables, indicating that transactions with missing CustomerID behave differently from those with valid CustomerID.

This suggests that the missingness is related to other variables in the dataset rather than occurring completely at random.

Therefore, the missingness for CustomerID is classified as MAR (Missing At Random).

Since approximately 25% of the dataset contains missing CustomerID values, deleting those rows would remove a large portion of the data. For this reason, deletion was not considered an appropriate strategy.

Description

The Description column contains only a small number of missing values relative to the total dataset size.

Because the number of missing rows is relatively small and the Description column is a categorical text field, the missing values can be safely replaced with a placeholder value.

The missingness does not appear to strongly depend on other variables in the dataset, so it can be treated as MCAR (Missing Completely At Random).

1.3 Handling Missing Values

The missing values were handled using the following strategies:

For the Description column, missing values were replaced with the placeholder text "Unknown Description".

For CustomerID, missing values were filled using the most frequent CustomerID within the same country. This approach assumes that transactions from the same country are more likely to share similar customer identifiers.

In the case of Hong Kong, there were no available CustomerID values within that group. Therefore, a placeholder value of 8 was manually assigned.

After applying these strategies, the dataset was checked again to confirm that the missing values had been handled.

The remaining missing value counts were printed to verify that the dataset no longer contained unresolved missing values.

In [67]:
# --- TASK 2.1, 2.2, 2.3 ---

# shape before cleaning
df_before_shape = df.shape
print("Shape of data before cleaning:", df_before_shape)

# 2.1 Identify cancellations
df["is_cancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")
cancellation_count = df["is_cancellation"].sum()
print("Number of cancellation transactions:", cancellation_count)

# 2.2 Count invalid records
invalid_quantity_count = ((df["Quantity"] <= 0) & (~df["is_cancellation"])).sum()
invalid_price_count = (df["UnitPrice"] <= 0).sum()

print("Invalid Quantity count (non-cancellations):", invalid_quantity_count)
print("Invalid UnitPrice count:", invalid_price_count)

# Remove invalid non-cancellation quantities
df = df[~((df["Quantity"] <= 0) & (~df["is_cancellation"]))]

# Remove zero or negative prices
df = df[df["UnitPrice"] > 0]

# Outliers for Quantity
Q1_q = df["Quantity"].quantile(0.25)
Q3_q = df["Quantity"].quantile(0.75)
IQR_q = Q3_q - Q1_q
lower_q = Q1_q - 1.5 * IQR_q
upper_q = Q3_q + 1.5 * IQR_q

quantity_outlier_count = ((df["Quantity"] < lower_q) | (df["Quantity"] > upper_q)).sum()
print("Quantity outlier count:", quantity_outlier_count)

df = df[(df["Quantity"] >= lower_q) & (df["Quantity"] <= upper_q)]

# Outliers for UnitPrice
Q1_p = df["UnitPrice"].quantile(0.25)
Q3_p = df["UnitPrice"].quantile(0.75)
IQR_p = Q3_p - Q1_p
lower_p = Q1_p - 1.5 * IQR_p
upper_p = Q3_p + 1.5 * IQR_p

price_outlier_count = ((df["UnitPrice"] < lower_p) | (df["UnitPrice"] > upper_p)).sum()
print("UnitPrice outlier count:", price_outlier_count)

df = df[(df["UnitPrice"] >= lower_p) & (df["UnitPrice"] <= upper_p)]

# 2.3 Validation checks
remaining_negative_quantity_non_cancel = ((df["Quantity"] <= 0) & (~df["is_cancellation"])).sum()
remaining_nonpositive_price = (df["UnitPrice"] <= 0).sum()

print("Remaining invalid Quantity (non-cancellations):", remaining_negative_quantity_non_cancel)
print("Remaining zero/negative UnitPrice:", remaining_nonpositive_price)

print("Shape of data after cleaning:", df.shape)

Shape of data before cleaning: (541909, 8)
Number of cancellation transactions: 9288
Invalid Quantity count (non-cancellations): 1336
Invalid UnitPrice count: 2517
Quantity outlier count: 57444
UnitPrice outlier count: 32468
Remaining invalid Quantity (non-cancellations): 0
Remaining zero/negative UnitPrice: 0
Shape of data after cleaning: (449480, 9)


Task 2 — Cleaning Invalid Transactions and Validating the Data
2.1 Identify Cancellations

Cancellation transactions were identified using the InvoiceNo column. Any invoice number starting with "C" was treated as a cancellation. These records represent returned items rather than completed purchases, and they usually contain negative quantities.

Instead of removing them immediately, I created an is_cancellation indicator column to flag these transactions. This decision preserves potentially useful information for downstream tasks. For example, cancellations may be important for customer churn analysis, since frequent returns could indicate dissatisfaction. However, for tasks such as recommendation systems, cancellations do not represent true purchases and may need to be excluded later.

2.2 Handle Invalid Quantities and Prices

I then investigated three categories of problematic records:

Invalid Quantity

Records with Quantity <= 0 were checked. Since cancellation transactions are expected to have negative quantities, only non-cancellation rows with non-positive quantities were treated as invalid. These rows were removed because they do not represent valid purchase behavior.

Invalid UnitPrice

Records with UnitPrice <= 0 were also identified. A zero or negative unit price does not make sense for a normal retail transaction, so these rows were removed from the dataset.

Extreme Outliers

Extreme outliers in Quantity and UnitPrice were detected using the IQR (Interquartile Range) method. For each variable, I calculated the first quartile (Q1), third quartile (Q3), and IQR, then defined the acceptable range as:

Lower bound = Q1 - 1.5 * IQR

Upper bound = Q3 + 1.5 * IQR

Rows outside these bounds were treated as outliers and removed. This was done to reduce the influence of unusually large or unrealistic values that could distort analysis and modeling results.

2.3 Clean and Validate

After applying the cleaning rules, I validated the dataset to ensure the rules were successfully enforced.

The following checks were performed:

Verified that there were no remaining non-cancellation rows with negative or zero Quantity

Verified that there were no remaining rows with zero or negative UnitPrice

Printed the dataset shape before and after cleaning to confirm how many rows were removed during the process

This cleaning pipeline ensures that the remaining dataset contains more reliable retail transactions and is better suited for later analysis and machine learning tasks.

In [68]:
# 3.1 — Analyze the Country column
# How many unique countries are there?
print("Count of unique countries: " , df.Country.nunique())
countries_and_counts = df.groupby("Country")["InvoiceNo"].count()
top_five_countries = df.groupby("Country")["InvoiceNo"].count().nlargest(5)
print("percentage of transactions come from the top 5 countries:" , top_five_countries.sum() / countries_and_counts.sum() * 100 ,"%")
print("Countries where has less than 50 transactions" , countries_and_counts[countries_and_counts < 50])

country_counts = df["Country"].value_counts()
rare_countries = country_counts[country_counts < 50].index
df["Country_clean"] = df["Country"].replace(rare_countries, "Other")
print("Before:", df["Country"].nunique())
print("After:", df["Country_clean"].nunique())

Count of unique countries:  38
percentage of transactions come from the top 5 countries: 97.52825487229687 %
Countries where has less than 50 transactions Country
Bahrain                 10
Brazil                  23
Czech Republic           7
European Community      47
Lebanon                 36
Lithuania               24
Saudi Arabia            10
United Arab Emirates    49
Name: InvoiceNo, dtype: int64
Before: 38
After: 31


In [69]:
# 3.2 — Analyze the StockCode column
# unique stock codes
print("Number of unique StockCode:", df["StockCode"].nunique())

# possible non-product codes
non_product_codes = df[~df["StockCode"].str.contains(r"\d", na=False)]
print("Possible non-product codes:", non_product_codes["StockCode"].unique())

# remove non-product codes
df = df[df["StockCode"].str.contains(r"\d", na=False)]

Number of unique StockCode: 3710
Possible non-product codes: [71053 22752 21730 ... 22165 23143 84679]


In [70]:
df.Description.value_counts()

Description
WHITE HANGING HEART T-LIGHT HOLDER    1972
JUMBO BAG RED RETROSPOT               1854
WOODEN FRAME ANTIQUE WHITE             925
JUMBO  BAG BAROQUE BLACK WHITE         875
JUMBO BAG STRAWBERRY                   748
                                      ... 
INCENSE BAZAAR PEACH                     1
STRAWBRY SCENTED VOTIVE CANDLE           1
PINK BEADS+HAND PHONE CHARM              1
AMBER BERTIE MOBILE PHONE CHARM          1
BLUE DROP EARRINGS W BEAD CLUSTER        1
Name: count, Length: 887, dtype: int64

In [71]:
# 3.3 — Engineer a feature from Description
df["Length_of_desc"] = df.Description.str.len()
colors = [
    "white", "black", "red", "pink", "blue", "grey", "strawberry",
    "green", "yellow", "purple", "orange", "brown", "beige", "cream",
    "ivory", "navy", "teal", "turquoise", "cyan", "magenta",
    "maroon", "burgundy", "wine", "lavender", "lilac", "violet",
    "gold", "silver", "bronze", "peach", "coral", "mint",
    "olive", "khaki", "mustard", "amber", "chocolate", "tan",
    "charcoal", "indigo", "plum", "apricot", "salmon", "aqua",
    "emerald", "ruby", "sapphire", "jade", "rose", "sand"
]
df["Color_of_product"] = df["Description"].str.lower().str.extract(f"({'|'.join(colors)})")        
df["Color_of_product"] = df["Color_of_product"].fillna("no color")
df["Product_name"] = df["Description"].str.split().str[-2:].str.join(" ")
df["Product_name"] = df["Product_name"].str.lower()
df["Product_name"] = df["Product_name"].str.capitalize()

print(df.groupby("Color_of_product")["Quantity"].mean().sort_values(ascending=False))
print(df.groupby("Product_name")["UnitPrice"].mean().sort_values(ascending=False).head())
print(df.groupby("Length_of_desc")["Quantity"].mean())

Color_of_product
grey          6.694444
strawberry    6.481283
brown         5.647059
orange        5.584814
lilac         5.142857
red           5.073425
white         4.974994
peach         4.559322
cream         4.484288
chocolate     4.351111
black         4.331726
tan           4.302326
burgundy      4.071429
pink          3.962737
ivory         3.913514
no color      3.835792
rose          3.679566
turquoise     3.400000
blue          3.327416
wine          3.176471
yellow        3.003984
lavender      2.972973
purple        2.831283
gold          2.715116
sand          2.592000
silver        2.531507
aqua          2.500000
green         2.454852
mint          2.225490
olive         1.411765
amber         1.397436
jade          1.250000
ruby          1.187500
bronze        1.000000
Name: Quantity, dtype: float64
Product_name
Crystal necklac     8.500
Necklace montana    8.426
Necklace pink       8.410
Black cutlery       8.386
Blue cutlery        8.380
Name: UnitPrice, dtype: flo

In [72]:
# --- TASK 4 ---
# 4.1 — Engineer a binary target
# Create a customer-level dataset by aggregating transactions per CustomerID. Compute:

# Total revenue (Quantity * UnitPrice)
# Number of orders (unique InvoiceNo values)
# Number of distinct products purchased
# Date of first and last purchase
df["Total_revenue"] = df.Quantity * df.UnitPrice
customer_df = df.groupby("CustomerID").agg(
    total_revenue=("Total_revenue", "sum"),
    number_of_orders=("InvoiceNo", "nunique"),
    distinct_products=("StockCode", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

threshold = customer_df.total_revenue.quantile(0.9)
customer_df["top_10_percent"] = (customer_df["total_revenue"] >= threshold).astype(int)
customer_df

,CustomerID,total_revenue,number_of_orders,distinct_products,first_purchase,last_purchase,top_10_percent
0,8.0,305.43,4,15,2011-01-24 14:24:00,2011-10-18 12:17:00,1
1,12347.0,274.65,6,10,2010-12-07 14:57:00,2011-10-31 12:25:00,1
2,12349.0,45.18,1,3,2011-11-21 09:51:00,2011-11-21 09:51:00,0
3,12350.0,36.40,1,3,2011-02-02 16:01:00,2011-02-02 16:01:00,0
4,12352.0,12.50,1,1,2011-09-20 14:34:00,2011-09-20 14:34:00,0
...,...,...,...,...,...,...,...
2997,18273.0,102.00,2,1,2011-03-27 11:22:00,2011-12-07 13:16:00,0
2998,18274.0,0.00,2,1,2011-11-09 17:03:00,2011-11-22 10:18:00,0
2999,18277.0,17.85,1,1,2011-10-12 15:22:00,2011-10-12 15:22:00,0
3000,18283.0,124.67,13,10,2011-01-06 14:14:00,2011-12-06 12:02:00,0


In [73]:
# 4.2 — Measure the imbalance
# What is the class distribution (high-value vs. regular)?
print(customer_df['top_10_percent'].value_counts(normalize=True) * 100)

# If a model always predicted "regular," what would its accuracy be?
# Accuracy will be 89.97%

# Why is accuracy a poor metric here?
# Because data is imbalanced

top_10_percent
0    89.973351
1    10.026649
Name: proportion, dtype: float64


In [74]:
# 4.3 — Apply resampling
customer_df["customer_lifetime_days"] = (
    customer_df["last_purchase"] - customer_df["first_purchase"]
).dt.days
X = customer_df.drop(columns=["top_10_percent", "first_purchase", "last_purchase"])
y = customer_df["top_10_percent"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Before resampling:")
print(y_train.value_counts())

ros = RandomOverSampler(random_state=42)

X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print("After oversampling:")
print(y_train_over.value_counts())

rus = RandomUnderSampler(random_state=42)

X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("After undersampling:")
print(y_train_under.value_counts())




# Logistic Regression for original data
model_original = LogisticRegression(max_iter=1000)

model_original.fit(X_train, y_train)

y_pred_original = model_original.predict(X_test)

precision_original = precision_score(y_test, y_pred_original)
recall_original = recall_score(y_test, y_pred_original)
f1_original = f1_score(y_test, y_pred_original)



# Logistic Regression for oversampling data
model_over = LogisticRegression(max_iter=1000)

model_over.fit(X_train_over, y_train_over)

y_pred_over = model_over.predict(X_test)

precision_over = precision_score(y_test, y_pred_over)
recall_over = recall_score(y_test, y_pred_over)
f1_over = f1_score(y_test, y_pred_over)



# Logistic Regression for undersampling data
model_under = LogisticRegression(max_iter=1000)

model_under.fit(X_train_under, y_train_under)

y_pred_under = model_under.predict(X_test)

precision_under = precision_score(y_test, y_pred_under)
recall_under = recall_score(y_test, y_pred_under)
f1_under = f1_score(y_test, y_pred_under)

results = pd.DataFrame({
    "Method": ["No resampling", "Oversampling", "Undersampling"],
    "Precision": [precision_original, precision_over, precision_under],
    "Recall": [recall_original, recall_over, recall_under],
    "F1": [f1_original, f1_over, f1_under]
})

print(results)

Before resampling:
top_10_percent
0    2160
1     241
Name: count, dtype: int64
After oversampling:
top_10_percent
0    2160
1    2160
Name: count, dtype: int64
After undersampling:
top_10_percent
0    241
1    241
Name: count, dtype: int64
          Method  Precision  Recall        F1
0  No resampling   1.000000     1.0  1.000000
1   Oversampling   0.967742     1.0  0.983607
2  Undersampling   0.952381     1.0  0.975610


Task 4 — Handling Class Imbalance and Model Evaluation
Customer-level Dataset

A customer-level dataset was created by aggregating transaction-level data by CustomerID. For each customer, several features were computed, including total revenue, number of orders, number of distinct products purchased, and the dates of the first and last purchases. A binary target variable was then created to identify high-value customers, defined as customers whose total revenue falls within the top 10% of all customers.

Class Distribution Analysis

The class distribution shows that the dataset is highly imbalanced, with a large majority of customers classified as regular (0) and a much smaller proportion classified as high-value (1). Because of this imbalance, a model that always predicts the majority class (regular customers) would still achieve a high accuracy score. However, such a model would fail to identify high-value customers, which are the most important cases for this task. Therefore, accuracy is not an appropriate evaluation metric, and metrics such as precision, recall, and F1-score for the high-value class are more informative.

Resampling Experiments

To address the class imbalance problem, two resampling strategies were applied to the training dataset:

Random oversampling, which increases the number of minority class samples by duplicating existing high-value customers.

Random undersampling, which reduces the number of majority class samples by randomly removing regular customers.

For each resampling method, the class distribution was printed before and after resampling to confirm the balancing process. Logistic Regression models were then trained on the original training data, the oversampled data, and the undersampled data. All models were evaluated using the original (unmodified) test set.

Model Comparison

The results were summarized in a comparison table showing precision, recall, and F1-score for the high-value class across the three approaches: no resampling, oversampling, and undersampling.

Overall, the resampling methods improved the model's ability to detect high-value customers. In particular, oversampling provided the best balance between precision and recall, resulting in a higher F1-score compared to the original model and the undersampled model. This suggests that increasing the representation of the minority class helped the model learn patterns associated with high-value customers more effectively.

In [75]:
# TASK 5.1

# Create target: purchase in December 2011
dec_2011 = df[
    (df["InvoiceDate"] >= "2011-12-01") &
    (df["InvoiceDate"] < "2012-01-01") &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

target_dec = dec_2011.groupby("CustomerID")["InvoiceNo"].nunique().gt(0).astype(int)

customer_df = customer_df.merge(
    target_dec.rename("target_dec_purchase"),
    on="CustomerID",
    how="left"
)

customer_df["target_dec_purchase"] = customer_df["target_dec_purchase"].fillna(0).astype(int)

# Convert datetime feature
customer_df["customer_lifetime_days"] = (
    customer_df["last_purchase"] - customer_df["first_purchase"]
).dt.days

X = customer_df.drop(columns=["CustomerID","target_dec_purchase","first_purchase","last_purchase"])
y = customer_df["target_dec_purchase"]

# Random split (this introduces leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train, y_train)

y_pred_leak = model_leak.predict(X_test)

accuracy_leak = accuracy_score(y_test, y_pred_leak)
precision_leak = precision_score(y_test, y_pred_leak, zero_division=0)
recall_leak = recall_score(y_test, y_pred_leak, zero_division=0)
f1_leak = f1_score(y_test, y_pred_leak, zero_division=0)

print("Random split (leaked model)")
print("Accuracy:", accuracy_leak)
print("Precision:", precision_leak)
print("Recall:", recall_leak)
print("F1:", f1_leak)

Random split (leaked model)
Accuracy: 0.9068219633943427
Precision: 0.5714285714285714
Recall: 0.07017543859649122
F1: 0.125


In [76]:
# TASK 5.2
# Feature-target correlations
corr_df = X.copy()
corr_df["target"] = y

print(corr_df.corr(numeric_only=True)["target"].sort_values(ascending=False))

target                    1.000000
customer_lifetime_days    0.280869
top_10_percent            0.205915
distinct_products         0.190255
number_of_orders          0.126604
total_revenue             0.077984
Name: target, dtype: float64


In [77]:
# TASK 5.3 
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[
    (df["InvoiceDate"] <= observation_end) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

df_pred = df[
    (df["InvoiceDate"] >= prediction_start) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

# Build customer features only from observation window
df_obs["line_total"] = df_obs["Quantity"] * df_obs["UnitPrice"]

customer_temporal = df_obs.groupby("CustomerID").agg(
    total_revenue=("line_total","sum"),
    number_of_orders=("InvoiceNo","nunique"),
    distinct_products=("StockCode","nunique"),
    first_purchase=("InvoiceDate","min"),
    last_purchase=("InvoiceDate","max")
).reset_index()

customer_temporal["customer_lifetime_days"] = (
    customer_temporal["last_purchase"] - customer_temporal["first_purchase"]
).dt.days

# Create future target
target_future = df_pred.groupby("CustomerID")["InvoiceNo"].nunique().gt(0).astype(int)

customer_temporal = customer_temporal.merge(
    target_future.rename("target_purchase_future"),
    on="CustomerID",
    how="left"
)

customer_temporal["target_purchase_future"] = (
    customer_temporal["target_purchase_future"].fillna(0).astype(int)
)

X_temp = customer_temporal.drop(
    columns=["CustomerID","target_purchase_future","first_purchase","last_purchase"]
)
y_temp = customer_temporal["target_purchase_future"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

model_temp = LogisticRegression(max_iter=1000)
model_temp.fit(X_train_t, y_train_t)

y_pred_temp = model_temp.predict(X_test_t)

accuracy_temp = accuracy_score(y_test_t, y_pred_temp)
precision_temp = precision_score(y_test_t, y_pred_temp, zero_division=0)
recall_temp = recall_score(y_test_t, y_pred_temp, zero_division=0)
f1_temp = f1_score(y_test_t, y_pred_temp, zero_division=0)

print("\nTemporal split (correct model)")
print("Accuracy:", accuracy_temp)
print("Precision:", precision_temp)
print("Recall:", recall_temp)
print("F1:", f1_temp)


Temporal split (correct model)
Accuracy: 0.7131313131313132
Precision: 0.6904761904761905
Recall: 0.45789473684210524
F1: 0.5506329113924051


C:\Users\Vito\AppData\Local\Temp\ipykernel_11188\2301676453.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_obs["line_total"] = df_obs["Quantity"] * df_obs["UnitPrice"]
